# ML Lab 8

**Aim:** Implement clustering algorithms to group data points based on similarity and evaluate clustering performance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

In [ ]:
# Load dataset
data = load_iris()
X = pd.DataFrame(data.data, columns=data.feature_names)
y_true = data.target

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('Dataset shape:', X.shape)
print('Features:', list(data.feature_names))
print('True class distribution:', np.bincount(y_true))

## Experiment 1: K-Means Clustering
1. Apply K-Means clustering on the dataset.
2. Visualize clusters using scatter plot.
3. Determine optimal number of clusters using Elbow Method.

In [ ]:
# Elbow Method
inertias = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# K-Means with k=3
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
km_labels = kmeans.fit_predict(X_scaled)

km_sil = silhouette_score(X_scaled, km_labels)
print('K-Means Silhouette Score:', round(km_sil, 4))

# Scatter plot (using first two features)
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=km_labels, cmap='viridis', s=50, alpha=0.7)
centers = kmeans.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, edgecolors='black', label='Centroids')
plt.xlabel(data.feature_names[0])
plt.ylabel(data.feature_names[1])
plt.title('K-Means Clustering (k=3)')
plt.colorbar(scatter, label='Cluster')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Experiment 2: Hierarchical Clustering
1. Apply Hierarchical (Agglomerative) clustering.
2. Plot dendrogram.
3. Form clusters and analyze them.

In [ ]:
# Dendrogram
linked = linkage(X_scaled, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(linked, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=10)
plt.title('Hierarchical Clustering Dendrogram (Ward Linkage)')
plt.xlabel('Sample Index / Cluster Size')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()

In [ ]:
# Agglomerative Clustering with 3 clusters
agg = AgglomerativeClustering(n_clusters=3, linkage='ward')
hc_labels = agg.fit_predict(X_scaled)

hc_sil = silhouette_score(X_scaled, hc_labels)
print('Hierarchical Clustering Silhouette Score:', round(hc_sil, 4))

# Scatter plot
plt.figure(figsize=(8, 6))
scatter = plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=hc_labels, cmap='viridis', s=50, alpha=0.7)
plt.xlabel(data.feature_names[0])
plt.ylabel(data.feature_names[1])
plt.title('Hierarchical Clustering (3 Clusters)')
plt.colorbar(scatter, label='Cluster')
plt.grid(alpha=0.3)
plt.show()

## Experiment 3: Compare K-Means vs Hierarchical Clustering
1. Visualize clusters side-by-side.
2. Analyze cluster separation and compactness.

In [ ]:
# Side-by-side cluster comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# K-Means
axes[0].scatter(X_scaled[:, 0], X_scaled[:, 1], c=km_labels, cmap='viridis', s=50, alpha=0.7)
axes[0].set_title(f'K-Means (Silhouette: {km_sil:.4f})')
axes[0].set_xlabel(data.feature_names[0])
axes[0].set_ylabel(data.feature_names[1])

# Hierarchical
axes[1].scatter(X_scaled[:, 0], X_scaled[:, 1], c=hc_labels, cmap='viridis', s=50, alpha=0.7)
axes[1].set_title(f'Hierarchical (Silhouette: {hc_sil:.4f})')
axes[1].set_xlabel(data.feature_names[0])
axes[1].set_ylabel(data.feature_names[1])

# True labels
axes[2].scatter(X_scaled[:, 0], X_scaled[:, 1], c=y_true, cmap='viridis', s=50, alpha=0.7)
axes[2].set_title('True Labels')
axes[2].set_xlabel(data.feature_names[0])
axes[2].set_ylabel(data.feature_names[1])

plt.tight_layout()
plt.show()

In [ ]:
# Comparison table
comparison_df = pd.DataFrame({
    'Method': ['K-Means', 'Hierarchical'],
    'Silhouette Score': [round(km_sil, 4), round(hc_sil, 4)],
})
print(comparison_df)